### **🔧 MCP Server — Research Assistant (ChromaDB + Firecrawl + Embeddings)**

This notebook implements the MCP server used by the LangGraph Research Assistant.
It exposes tools for:

- Saving research data into topic‑scoped Chroma vectorstores  
- Semantic retrieval using local Ollama embeddings  
- Topic management (list, delete, inspect)  
- Firecrawl web scraping and crawling  

> ⚠️ **Important:**  
> MCP servers must run as standalone processes using `stdio`.  
> This notebook is for *understanding the server code*, not for hosting it.  

##### 🔍 Check Local Ollama Models
- Quick sanity check to confirm Ollama is running and models are available.
- Edit your Ollama base url — `OLLAMA_BASE_URL` in `.env`, e.g. http://localhost:11434.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")

!curl {OLLAMA_BASE_URL}/api/tags

{"models":[{"name":"nomic-embed-text:latest","model":"nomic-embed-text:latest","modified_at":"2026-04-20T03:42:06.9664384+08:00","size":274302450,"digest":"0a109f422b47e3a30ba2b10eca18548e944e8a23073ee3f3e947efcf3c45e59f","details":{"parent_model":"","format":"gguf","family":"nomic-bert","families":["nomic-bert"],"parameter_size":"137M","quantization_level":"F16"}},{"name":"llama3:latest","model":"llama3:latest","modified_at":"2026-04-10T18:28:54.6963824+08:00","size":4661224676,"digest":"365c0bd3c000a25d28ddbf732fe1c6add414de7275464c4e4d1c3b5fcb5d8ad1","details":{"parent_model":"","format":"gguf","family":"llama","families":["llama"],"parameter_size":"8.0B","quantization_level":"Q4_0"}},{"name":"qwen3:latest","model":"qwen3:latest","modified_at":"2026-04-07T03:42:05.539214+08:00","size":5225388164,"digest":"500a1f067a9f782620b40bee6f7b0c89e17ae61f686b92c24933e4ca4b2b8b41","details":{"parent_model":"","format":"gguf","family":"qwen3","families":["qwen3"],"parameter_size":"8.2B","quanti

#### 🧪 Test Embedding Model
Verify that the embedding model (`nomic-embed-text`) is working correctly.

In [2]:
import requests
requests.post(f"{OLLAMA_BASE_URL}/api/embed", json={"model": "nomic-embed-text", "input": ["hello"]}).json()

{'model': 'nomic-embed-text',
 'embeddings': [[0.017890694,
   -0.005857372,
   -0.17527784,
   -0.013715897,
   0.03402557,
   0.044756383,
   0.012405766,
   -0.0025463272,
   -0.0147743,
   -0.039300237,
   -0.009206238,
   0.05171858,
   0.05771968,
   0.057204816,
   0.04501635,
   -0.053590104,
   0.028793193,
   -0.047167517,
   -0.039059866,
   0.026980367,
   0.009358973,
   -0.06673761,
   0.004942086,
   -0.0059101107,
   0.17225455,
   -0.004390079,
   0.018655416,
   0.08340848,
   0.0016043204,
   -0.022531617,
   0.020353707,
   -0.018748302,
   0.015327549,
   0.0032236974,
   0.027561733,
   -0.018714128,
   -0.0038400358,
   0.011211327,
   0.016209979,
   0.019288898,
   -0.017633036,
   -0.009611144,
   0.009241481,
   -0.028306039,
   0.05137155,
   0.0072837393,
   -0.022140808,
   0.013336251,
   0.053701054,
   -0.043478094,
   -0.0039904388,
   -0.0042095003,
   -0.02214747,
   0.04244085,
   0.033540893,
   0.0549708,
   0.057884112,
   0.020398319,
   0.02176

#### 📦 Server Dependencies
Core imports for building the MCP server, Chroma vectorstore, and embedding pipeline.

In [3]:
from mcp.server.fastmcp import FastMCP
from pathlib import Path
from typing import List, Set, Dict, Any
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings
from langchain_core.documents import Document
import shutil
import hashlib
import json

#### 🚀 Initialize MCP Server
Create the FastMCP server instance and configure paths, embedding model, and ChromaDB root.

In [4]:
# Initialize MCP Server
mcp = FastMCP("Research Assistant")

# Constants
parent_dir = Path.cwd().parent
CHROMA_DB_ROOT = parent_dir / "research_chroma_dbs"
EMBED_MODEL = "nomic-embed-text"
#OLLAMA_BASE_URL = "http://localhost:11434"
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL")

# Initialize embeddings once
embeddings = OllamaEmbeddings(
    model=EMBED_MODEL,
    base_url=OLLAMA_BASE_URL
)

#### 🧰 Utility Functions
Helper functions for hashing content, loading metadata, and constructing vectorstores.
These support deduplication and topic‑scoped storage.

In [5]:
def get_content_hash(content: str) -> str:
    """Generate a hash for content to check for duplicates."""
    return hashlib.md5(content.encode('utf-8')).hexdigest()

def load_content_hashes(topic_path: Path) -> Set[str]:
    """Load existing content hashes from metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    if metadata_file.exists():
        try:
            with open(metadata_file, 'r') as f:
                return set(json.load(f))
        except:
            return set()
    return set()

def save_content_hashes(topic_path: Path, hashes: Set[str]):
    """Save content hashes to metadata file."""
    metadata_file = topic_path / "content_hashes.json"
    with open(metadata_file, 'w') as f:
        json.dump(list(hashes), f)

def get_vectorstore(topic: str) -> Chroma:
    """Get or create a ChromaDB vectorstore for a topic."""
    topic_path = CHROMA_DB_ROOT / topic
    topic_path.mkdir(parents=True, exist_ok=True)
    
    return Chroma(
        persist_directory=str(topic_path),
        embedding_function=embeddings,
        collection_name=f"research_{topic}"
    )

#### 🛠️ MCP Tools Implementation
Each function below becomes an MCP tool exposed to the LangGraph client.

Tools include:
- Saving research data  
- Semantic search  
- Topic listing & deletion  
- Get detailed topic information

In [6]:
# === Tools ===

@mcp.tool()
def save_research_data(content: List[str], topic: str = "default") -> str:
    """
    Save research content to vector database for future retrieval.
    Args:
        content: List of text content to save
        topic: Topic name for organizing the data (creates separate DB)
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic
        topic_path.mkdir(parents=True, exist_ok=True)
        
        # Load existing content hashes
        existing_hashes = load_content_hashes(topic_path)
        
        # Filter out duplicate content
        new_content = []
        new_hashes = set(existing_hashes)
        
        for text in content:
            content_hash = get_content_hash(text)
            # if content_hash not in existing_hashes:
            if content_hash not in new_hashes:
                new_content.append(text)
                new_hashes.add(content_hash)
        
        if not new_content:
            return f"No new content to save - all {len(content)} documents already exist in topic: {topic}"
        
        # Get vectorstore for this topic
        vectorstore = get_vectorstore(topic)
        
        # Create documents with metadata
        documents = []
        doc_ids = []
        for i, text in enumerate(new_content):
            content_hash = get_content_hash(text)
            doc = Document(
                page_content=text,
                metadata={
                    "topic": topic,
                    "content_hash": content_hash,
                    "doc_index": len(existing_hashes) + i
                }
            )
            documents.append(doc)
            doc_ids.append(f"{topic}_{content_hash}")
        
        # Add documents to vectorstore
        vectorstore.add_documents(documents=documents, ids=doc_ids)
        
        # Save updated content hashes
        save_content_hashes(topic_path, new_hashes)
        
        return f"Successfully saved {len(new_content)} new documents to topic: {topic} (skipped {len(content) - len(new_content)} duplicates)"
        
    except Exception as e:
        return f"Error saving research data: {str(e)}"

@mcp.tool()
def search_research_data(query: str, topic: str = "default", max_results: int = 5) -> str:
    """
    Search through saved research data using semantic similarity.
    Args:
        query: Search query
        topic: Topic database to search in
        max_results: Maximum number of results to return
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic
        
        if not topic_path.exists():
            return f"No research data found for topic: {topic}"
        
        # Get vectorstore for this topic
        vectorstore = get_vectorstore(topic)
        
        # Check if collection has any documents
        try:
            collection = vectorstore._collection
            count = collection.count()
            if count == 0:
                return f"No documents found in topic: {topic}"
        except:
            return f"No research data found for topic: {topic}"
        
        # Search for similar documents
        results = vectorstore.similarity_search_with_score(query, k=max_results)
        
        if not results:
            return f"No relevant results found for query: '{query}' in topic: {topic}"
        
        # Format results
        formatted_results = []
        for i, (doc, score) in enumerate(results):
            similarity = 1 - score  # Convert distance to similarity
            result_text = f"Result {i+1} (Similarity: {similarity:.3f}):\n{doc.page_content}\n"
            formatted_results.append(result_text)
        
        return "\n" + "="*50 + "\n".join(formatted_results) + "="*50
        
    except Exception as e:
        return f"Error searching research data: {str(e)}"

@mcp.tool()
def list_research_topics() -> str:
    """
    List all available research topics (vector databases).
    """
    try:
        if not CHROMA_DB_ROOT.exists():
            return "No research topics found"
        
        topics = []
        for path in CHROMA_DB_ROOT.iterdir():
            if path.is_dir():
                # Try to get document count from ChromaDB
                try:
                    vectorstore = get_vectorstore(path.name)
                    collection = vectorstore._collection
                    doc_count = collection.count()
                    topics.append(f"Topic: {path.name} ({doc_count} documents)")
                except Exception as e:
                    # Fallback to hash count if ChromaDB fails
                    try:
                        hashes = load_content_hashes(path)
                        doc_count = len(hashes)
                        topics.append(f"Topic: {path.name} ({doc_count} documents)")
                    except:
                        topics.append(f"Topic: {path.name}")
        
        if not topics:
            return "No research topics found"
        
        return "\n".join(topics)
        
    except Exception as e:
        return f"Error listing topics: {str(e)}"

@mcp.tool()
def delete_research_topic(topic: str) -> str:
    """
    Delete a research topic and all its data.
    Args:
        topic: Topic name to delete
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic
        
        if not topic_path.exists():
            return f"Topic '{topic}' does not exist"
        
        # Try to delete the ChromaDB collection first
        try:
            vectorstore = get_vectorstore(topic)
            vectorstore.delete_collection()
        except:
            pass  # Continue even if collection deletion fails
        
        # Remove the entire directory
        shutil.rmtree(topic_path)
        
        return f"Successfully deleted topic: {topic}"
        
    except Exception as e:
        return f"Error deleting topic: {str(e)}"

@mcp.tool()
def get_topic_info(topic: str) -> str:
    """
    Get detailed information about a research topic.
    Args:
        topic: Topic name to get info for
    """
    try:
        topic_path = CHROMA_DB_ROOT / topic
        
        if not topic_path.exists():
            return f"Topic '{topic}' does not exist"
        
        # Get vectorstore info
        vectorstore = get_vectorstore(topic)
        collection = vectorstore._collection
        doc_count = collection.count()
        
        # Get content hashes info
        hashes = load_content_hashes(topic_path)
        hash_count = len(hashes)
        
        info = f"""Topic Information: {topic}
                - ChromaDB Collection: research_{topic}
                - Document Count: {doc_count}
                - Hash Records: {hash_count}
                - Database Path: {topic_path}
                - Embedding Model: {EMBED_MODEL}
                - Ollama URL: {OLLAMA_BASE_URL}"""
        
        return info
        
    except Exception as e:
        return f"Error getting topic info: {str(e)}"


##### **save_research_data**
Stores new (deduplicated) text chunks into a topic‑specific Chroma vectorstore.

In [7]:
save_research_data(["hello"], topic="default")

[04/26/26 15:52:42] INFO     HTTP Request: POST http://192.168.89.222:11434/api/embed "HTTP/1.1 200 ]8;id=14009292;file:///mnt/c/Users/User/Udemy/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/mcp-langgraph/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14009293;file:///mnt/c/Users/User/Udemy/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/mcp-langgraph/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

'Successfully saved 1 new documents to topic: default (skipped 0 duplicates)'

##### **search_research_data**
Performs semantic similarity search within a topic’s vectorstore.

In [8]:
search_research_data("hello", topic="default")

[04/26/26 15:52:46] INFO     HTTP Request: POST http://192.168.89.222:11434/api/embed "HTTP/1.1 200 ]8;id=14009298;file:///mnt/c/Users/User/Udemy/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/mcp-langgraph/.venv/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=14009299;file:///mnt/c/Users/User/Udemy/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/mcp-langgraph/.venv/lib/python3.13/site-packages/httpx/_client.py#1025\1025]8;;\
                             OK"                                                                                   

'\n==================================================Result 1 (Similarity: 1.000):\nhello\n=================================================='

##### **list_research_topics**
Returns all available research topics stored on disk.

In [9]:
list_research_topics()

'Topic: Amazon-Stock-Analysis (5 documents)\nTopic: default (1 documents)\nTopic: Google-Stock-Analysis (5 documents)\nTopic: Microsoft-Stock-Analysis (5 documents)'

##### **get_topic_info**
Get detailed information about a research topic.

In [10]:
print(get_topic_info("default"))

Topic Information: default
                - ChromaDB Collection: research_default
                - Document Count: 1
                - Hash Records: 1
                - Database Path: /mnt/c/Users/User/Udemy/mcp-mastery-build-ai-apps-with-claude-langchain-and-ollama/mcp-langgraph/research_chroma_dbs/default
                - Embedding Model: nomic-embed-text
                - Ollama URL: http://192.168.89.222:11434


##### **delete_research_topic**
Deletes an entire topic directory and its associated vectorstore.

In [11]:
delete_research_topic("default")

'Successfully deleted topic: default'

#### 🌐 Firecrawl Tools
These tools provide web scraping, crawling, extraction, and browser automation capabilities.
They are forwarded directly to the MCP client.

- **firecrawl_search**  
  Searches the web and returns structured results.

- **firecrawl_scrape**  
  Scrapes a single URL and extracts cleaned content.

- **firecrawl_map**  
  Maps and extracts multiple URLs in batch.

- **firecrawl_crawl**  
  Starts a full‑site crawl job.

- **firecrawl_check_crawl_status**  
  Checks the status of a running crawl job.

- **firecrawl_extract**  
  Extracts structured data from a webpage.

- **firecrawl_agent**  
  Runs Firecrawl’s autonomous crawling agent.

- **firecrawl_agent_status**  
  Checks the status of an agent‑based crawl.

- **firecrawl_browser_create**  
  Creates a headless browser session.

- **firecrawl_browser_execute**  
  Executes browser actions (navigate, click, extract, etc.).

- **firecrawl_browser_delete**  
  Deletes a browser session.

- **firecrawl_browser_list**  
  Lists active browser sessions.

### **▶️ Start MCP Server (real execution)**

The MCP server must run using `stdio` transport, which Jupyter Notebook cannot provide.

To launch the server properly, start it from a terminal instead:

```bash
python server.py

In [ ]:
# This call is required when running the MCP server as a standalone script.
# It is intentionally not executed inside the notebook environment.
mcp.run(transport="stdio")